# vLLM hands-on intro (Google Colab, free T4 GPU)

**Goal:** see what an inference engine does, using a real NVIDIA GPU, in about an hour.

You will:
1. Install vLLM and load a small model (Qwen2.5-1.5B-Instruct)
2. Read the startup logs: **KV cache size** and **max concurrency** (PagedAttention at work)
3. Compare **one-at-a-time vs batched** generation: why vLLM exists
4. See **prefix caching** reuse shared prompt work
5. Run vLLM as an **OpenAI-compatible server**, fire concurrent requests, and watch the **scheduler metrics**

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the cells top to bottom.

## 0. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
# Expect: Tesla T4, ~15 GB, compute capability 7.5
# The T4 has no bfloat16 support, so we use dtype="half" (float16) everywhere.

## 1. Install vLLM
Takes about 2–4 minutes. It installs a matching PyTorch build too. If Colab shows a
"restart session" prompt afterwards, accept it and continue from the next cell.

In [ ]:
!pip install -q vllm
import vllm, torch
print("vLLM", vllm.__version__, "| torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Load a model, then read the logs
While it loads, look in the output for lines like:

- `Model loading took X GiB`: memory for the **weights** (~3 GiB for 1.5B params in fp16)
- `... KV cache size: N tokens, Maximum concurrency for 4096 tokens per request: Yx`: memory left over for the **KV cache** (the model's "notes" for every token), and how many full-length requests fit **at once**

`gpu_memory_utilization=0.85` tells vLLM: "take 85% of the GPU; weights first, the rest becomes KV-cache blocks".
That pre-allocated pool of blocks is what PagedAttention hands out on demand.

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    dtype="half",
    max_model_len=4096,
    gpu_memory_utilization=0.85,
)

## 3. First generation

In [ ]:
params = SamplingParams(temperature=0.7, max_tokens=128)
out = llm.chat([{"role": "user", "content": "Explain a KV cache to a software engineer in 3 sentences."}], params)
print(out[0].outputs[0].text)

## 4. The key experiment: one-at-a-time vs batched
Same 64 prompts, two ways:
- **Sequential:** send one, wait, send the next (like a naive server)
- **Batched:** give vLLM all 64 at once; its scheduler runs them together (continuous batching)

Watch the **tokens/sec** difference.

In [ ]:
import time

topics = ["Python", "Rust", "Kubernetes", "TCP", "B-trees", "GPUs", "Docker", "HTTP/2",
          "Redis", "Kafka", "OAuth", "DNS", "Git", "Linux", "SQL indexes", "WebSockets"]
prompts = [[{"role": "user", "content": f"Give one practical tip about {t} (variation {i})."}]
           for i in range(4) for t in topics]   # 64 prompts
params = SamplingParams(temperature=0.8, max_tokens=96)

def count_tokens(outputs):
    return sum(len(o.outputs[0].token_ids) for o in outputs)

# Sequential (first 16 only, to save time; we scale up the estimate)
t0 = time.time(); seq_tokens = 0
for p in prompts[:16]:
    seq_tokens += count_tokens(llm.chat([p], params, use_tqdm=False))
seq_time = time.time() - t0

# Batched (all 64 in one call)
t0 = time.time()
batch_out = llm.chat(prompts, params, use_tqdm=False)
batch_time = time.time() - t0; batch_tokens = count_tokens(batch_out)

print(f"Sequential: {seq_tokens/seq_time:7.1f} tokens/sec  (16 prompts in {seq_time:.1f}s → ~{seq_time*4:.0f}s for 64)")
print(f"Batched   : {batch_tokens/batch_time:7.1f} tokens/sec  (64 prompts in {batch_time:.1f}s)")
print(f"Speed-up  : ~{(batch_tokens/batch_time)/(seq_tokens/seq_time):.1f}x on the SAME GPU")

**Why?** Each generation step has to read *all* the model weights from GPU memory. For one request,
the GPU mostly waits on memory. With 64 requests, one read of the weights serves 64 tokens.
Batching turns idle GPU time into throughput, and PagedAttention is what lets many requests' KV caches fit at once.

## 5. Prefix caching
Many requests share the same long start (a system prompt, or a document in RAG).
vLLM caches the KV blocks for that shared prefix and reuses them instead of recomputing.

In [ ]:
long_system = "You are a senior SRE. Follow these runbook rules carefully. " * 150  # ~1.5k tokens shared prefix
qs = ["How do I debug high latency?", "What should I check first in an outage?", "How do I roll back safely?"]
params = SamplingParams(temperature=0, max_tokens=1)  # 1 token → we mostly time the prompt processing

for round_ in ["cold (prefix computed)", "warm (prefix reused)"]:
    t0 = time.time()
    llm.chat([[{"role": "system", "content": long_system}, {"role": "user", "content": q}] for q in qs],
             params, use_tqdm=False)
    print(f"{round_:25s}: {time.time()-t0:.3f}s")

## 6. Run vLLM as an OpenAI-compatible server
This is how vLLM runs in production. **First free the GPU:** `Runtime → Restart session`,
then run the cells below (they don't depend on anything above).

In [ ]:
import subprocess, time, requests

server = subprocess.Popen(
    "vllm serve Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 "
    "--gpu-memory-utilization 0.85 --port 8000 > vllm_server.log 2>&1",
    shell=True,
)
for _ in range(120):                      # wait up to ~10 min for startup
    try:
        if requests.get("http://localhost:8000/health", timeout=2).ok:
            print("Server is up ✅"); break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(5)
else:
    print("Server did not start; see the log below")
!tail -n 15 vllm_server.log

Any OpenAI client works by just changing `base_url`, which is why vLLM is easy to adopt:

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
r = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": "In one line: what is continuous batching?"}],
    max_tokens=60,
)
print(r.choices[0].message.content)
print("usage:", r.usage)

### Watch the scheduler under load
We fire 48 requests concurrently and sample vLLM's Prometheus metrics while they run:
- `num_requests_running`: requests in the current batch
- `num_requests_waiting`: queued because the batch or KV cache is full
- `kv_cache_usage_perc` (named `gpu_cache_usage_perc` in older versions): how full the KV-cache block pool is

In [ ]:
import threading, re

def ask(i):
    client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct",
        messages=[{"role": "user", "content": f"Write a detailed 300-word story about robot #{i}."}],
        max_tokens=400,
    )

def metrics():
    text = requests.get("http://localhost:8000/metrics").text
    pick = {}
    for line in text.splitlines():
        m = re.match(r"vllm:(num_requests_running|num_requests_waiting|kv_cache_usage_perc|gpu_cache_usage_perc)\{.*\} ([0-9.e+-]+)", line)
        if m: pick[m.group(1)] = float(m.group(2))
    return pick

threads = [threading.Thread(target=ask, args=(i,)) for i in range(48)]
for t in threads: t.start()
while any(t.is_alive() for t in threads):
    print(metrics()); time.sleep(2)
for t in threads: t.join()
print("done:", metrics())

Also look at the server's own periodic stats line (throughput, running and waiting counts, KV cache usage):

In [ ]:
!grep -E "throughput|Running|KV cache" vllm_server.log | tail -n 10

In [ ]:
server.terminate()   # stop the server when you're done

## 7. Try these yourself
1. Change `gpu_memory_utilization` to `0.5`. How do the KV cache size and max concurrency change? Do more requests end up **waiting**?
2. Set `max_model_len=1024` vs `8192`. What happens to max concurrency, and why?
3. Raise the concurrent requests from 48 to 200. Watch `num_requests_waiting`.
4. Try a quantized model, e.g. `Qwen/Qwen2.5-7B-Instruct-AWQ` (4-bit). A 7B model that wouldn't fit in fp16 now fits.

## How this connects to the vLLM code (on your Mac: `03-vllm-contrib/vllm/`)
| What you saw | Where it lives |
|---|---|
| OpenAI API server | `vllm/entrypoints/openai/` |
| Running/waiting queues | `vllm/v1/core/sched/scheduler.py` |
| KV cache blocks and prefix caching | `vllm/v1/core/kv_cache_manager.py`, `vllm/v1/core/block_pool.py` |
| The Qwen2 model math | Qwen2 now runs via the Hugging Face Transformers backend: `vllm/model_executor/models/transformers/` (see `registry.py`). For a native vLLM model, compare `vllm/model_executor/models/llama.py` |
| `KV cache size` / `Maximum concurrency` log line | `vllm/v1/core/kv_cache_utils.py` |
| `/metrics` | `vllm/v1/metrics/` |

**Save your copy:** `File → Save a copy in Drive`, or Colab's session will discard your changes.